# 07: PersistentVolume, ConfigMap & Secrets
Full hands-on: Frontend → Backend (fake JSON) → Postgres (persistent) → Rolling Update

## 1. Cluster Setup
```bash
kind create cluster --name k8-lab
kubectl create namespace dev
```
> **Oops:** `kind create namespace dev` ✗ — `kind` doesn't create namespaces. Use `kubectl create namespace`.

## 2. Frontend
```bash
cd 07.pv,configmap,secrers/frontend
docker build -t frontend-07:1.0 .
kind load docker-image frontend-07:1.0 --name k8-lab
kubectl apply -f frontend.yaml
kubectl get pods -n dev  # 3/3 Running
```
> **Oops:** Ran `docker build` from root first — no Dockerfile. `cd` to correct dir first.

## 3. Backend v1.0 — Fake JSON

```python
# backend/main.py — hardcoded mock data
from fastapi import FastAPI
app = FastAPI()

@app.get("/")
def home():
    users = [
        {"id": 1, "name": "Alice"},
        {"id": 2, "name": "Bob"},
        {"id": 3, "name": "Charlie"}
    ]
    return {"users": users}
```

```bash
cd 07.pv,configmap,secrers/backend
docker build -t backend-07:1.0 .
kind load docker-image backend-07:1.0 --name k8-lab
kubectl apply -f backend.yaml
kubectl port-forward service/backend 8080:80 -n dev
# → {"users":[{"id":1,"name":"Alice"},...]}
```

**Pattern:** Mock data first → real DB later. API contract stays unchanged.

## 4. ConfigMap & Secret — Theory

### ConfigMap
- Stores non-sensitive config: `DB_HOST=postgres`, `LOG_LEVEL=INFO`
- Usually safe to commit to Git

### Secret
- Stores sensitive data: passwords, API keys, tokens
- **Base64 is encoding, NOT encryption** — instant decode:
  ```bash
  echo "bXlwYXNzd29yZA==" | base64 -d  # → mypassword
  ```
- Never commit real Secret values to Git

### Production Pattern
```
GitHub → CI/CD → Secrets Manager → K8s Secret (never in repo)
```
Commit `secret.example.yaml` (template) instead.

### Rule of Thumb
| Object | Content | Commit? |
|--------|---------|---------|
| ConfigMap | Non-sensitive | Usually yes |
| Secret | Passwords/keys/tokens | Never real values |
| `secret.example.yaml` | Placeholder template | Yes |
| Actual Secret | Real credentials | No — deploy-time only |

**For this project:** `password123` is fine — local kind cluster, learning only.

## 5. PostgreSQL — Persistent Storage

### 5a. Secret
```bash
kubectl apply -f postgres/secrets.yaml
```
Stores `POSTGRES_USER` + `POSTGRES_PASSWORD`. Injected into StatefulSet via `secretKeyRef`.

### 5b. PVC
```bash
kubectl apply -f postgres/pvc.yaml
```
Requests 1Gi `ReadWriteOnce` storage. Initially `Pending` — normal until claimed by a Pod.

### 5c. StatefulSet — the tricky one
```bash
kubectl apply -f postgres/statefulset.yaml
```

> **CRITICAL ERROR:** ❌
> ```
> Error: unknown field "spec.template.spec.containers[0].volumes"
> ```
> **Why?** In StatefulSet, persistent storage uses `volumeClaimTemplates` at `spec` level — **NOT** `volumes:` inside the container spec (that's a Deployment thing).

**Key StatefulSet features:**
- Pods named `postgres-0`, `postgres-1`, etc. (stable identity)
- Each Pod gets its own PVC automatically
- Ordered creation/termination
- Works with headless service for direct Pod DNS

### 5d. Headless Service
```bash
kubectl apply -f postgres/service.yaml
```
`clusterIP: None` → enables `postgres-0.postgres.dev.svc.cluster.local` DNS.

### Verify
```bash
kubectl get pods -n dev
# postgres-0   1/1   Running
kubectl get pvc -n dev
# postgres-storage-postgres-0   Bound
```

## 6. Backend v1.1 — Real Postgres

Updated `main.py` to read from **environment variables** (not hardcoded):

```python
from fastapi import FastAPI
import psycopg2
import os

app = FastAPI()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),      # ConfigMap
    database=os.getenv("DB_NAME"),  # ConfigMap
    user=os.getenv("DB_USER"),      # Secret
    password=os.getenv("DB_PASSWORD"), # Secret
    port=os.getenv("DB_PORT", "5432") # ConfigMap
)

@app.get("/")
def get_users():
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users")
    rows = cursor.fetchall()
    cursor.close()
    return {"users": rows}
```

**Config source:**
```
ConfigMap            Secret
   │                    │
DB_HOST              DB_USER
DB_NAME              DB_PASSWORD
DB_PORT
   │                    │
   └──── os.getenv() ───┘
            │
         Backend (never knows origin)
```

```bash
docker build -t backend-07:1.1 .
kind load docker-image backend-07:1.1 --name k8-lab
kubectl apply -f backend.yaml
kubectl rollout status deployment/backend-deployment -n dev
```

## 7. Rolling Update

Changed image tag `backend-07:1.0` → `backend-07:1.1` in `backend.yaml`.

```
Old Pod (v1.0)  →  New Pod (v1.1) starts  →  Old Pod drains & stops
                      ↓
              Repeat for all 3 replicas
```

- Zero-downtime: new Pods start before old ones terminate
- `kubectl rollout status` tracks progress
- Backend never stops serving

## 8. Final Architecture

```
Browser
   │
   ▼
frontend-service (ClusterIP)
   │
   ▼
Frontend Pods (3)
   │  HTTP GET /users
   ▼
backend-service (ClusterIP)
   │
   ▼
FastAPI Pods (3)
   │  Reads ConfigMap & Secret (env vars)
   │  psycopg2 connection
   ▼
postgres-service (headless, clusterIP: None)
   │
   ▼
PostgreSQL StatefulSet (postgres-0)
   │
   ▼
PVC (postgres-storage-postgres-0)
   │
   ▼
PV (kind host — local storage)
```

## Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **ConfigMap** | Non-sensitive config in env vars — usually safe to commit |
| **Secret** | Base64 ≠ encryption. Never commit real values. Use `secret.example.yaml` |
| **StatefulSet** | Stable identity (`postgres-0`), ordered creation, `volumeClaimTemplates` (NOT `volumes:` in container) |
| **PVC** | Storage survives Pod restarts — decoupled from Pod lifecycle |
| **Headless Service** | `clusterIP: None` enables direct Pod DNS for StatefulSet |
| **Rolling Update** | Zero-downtime — change image tag, let K8s handle the rest |
| **Mock → Real** | Start with fake JSON, swap to DB later — API contract stays same |

## For the Final Project
We will do the production deployment manually (not committed). Actual `secret.yaml` will be generated at deploy time from a secrets manager.